pip install anndata rpy2 pandas

Ensure R packages like data.table, dplyr, etc., are installed if you use them in your R script.

In [ ]:
import anndata
import numpy as np
import pandas as pd
from rpy2 import robjects
from rpy2.robjects import pandas2ri
from rpy2.robjects.packages import importr

# Activate pandas-to-R dataframe conversion
pandas2ri.activate()

# Load your AnnData
adata = anndata.read_h5ad("your_file.h5ad")

# Convert the .X matrix to a pandas DataFrame (ensure it's dense)
X_df = pd.DataFrame(adata.X.toarray() if not isinstance(adata.X, np.ndarray) else adata.X,
                    index=adata.obs_names,
                    columns=adata.var_names)

# Pass it to R
robjects.globalenv["X_df"] = pandas2ri.py2rpy(X_df)

In [ ]:
robjects.r('''
# Example R processing
df1 <- head(X_df)
df2 <- tail(X_df)
df3 <- data.frame(summary=rowMeans(X_df))

# Bundle them into a list to return
result_list <- list(df1 = df1, df2 = df2, df3 = df3)
''')


In [ ]:
result = robjects.globalenv['result_list']
df1 = pandas2ri.rpy2py(result.rx2('df1'))
df2 = pandas2ri.rpy2py(result.rx2('df2'))
df3 = pandas2ri.rpy2py(result.rx2('df3'))

In [ ]:
adata.uns['df1'] = df1
adata.uns['df2'] = df2
adata.uns['df3'] = df3

# Save modified file
adata.write("modified_file.h5ad")